Решил попробовать матричную факторизацию и поменять подход работы с данными, потому что знал что у нас не будет user-item полной матрицы(Ну тут немного конечно нечестно, потому что изначально я планировал и KNN-ов всякие метрики посмотреть, но там оч сильно не вместилось), поэтому:

1. Решил оставить взаимодействия полученные с помощью река яндекса с кэфом 0.8
2. Сделал систему с весами у типов событий не приводил ее к [0, 1]
3. Решил попробовать time decay идея мне нравится, потому что чисто эвристически мы действительно слушаем музыку "сезонами".TODO Но тут надо как то совмещать, чтобы были какие-то еще древние реки наверное которые
4. Сделал по прослушиваниям настройку, так как есть процентный признак по сути тот же Listen+, так как использовал на 0.5, также у прослушивания простов вес меньше чем у лайка, но в теории можно попробовать будет придумать какую то функцию где прослушивание почти полное дает сильно больше веса чем например 0.6 пока оставил просто линейно
5. Прологорифмировал, потому что с нормализацией, что-то очень маленькие веса получались

В целом получились хорошие метрики лучше, чем без рек событий, но KNN-ны Богдана, побить не получилось

In [5]:
import pandas as pd

from get_data import prepare
from models import iALS, WeightedALS, BPR
from metrics import recall_at_k, precision_at_k, ndcg_at_k

In [2]:
res = prepare(
    path="yambda",
    test_days=14,
    q1=0.7,
    q2=0.95,
    organic_k=0.8,
    decay_days=90.0,
    listen_k=0.9,
)

user_item_matrix = res["user_item_matrix"]
user2id = res["user2id"]
id2item = res["id2item"]
test_true = res["test_true"]

print("matrix shape:", user_item_matrix.shape)
print("nnz:", user_item_matrix.nnz)
print("test users:", len(test_true))

matrix shape: (2496, 470449)
nnz: 4212276
test users: 2490


In [3]:
def eval_model(model, name, ks=(10, 20)):
    model.fit(user_item_matrix)

    res = {"model": name}

    for k in ks:
        recall = recall_at_k(
            test_true=test_true,
            user_item_matrix=user_item_matrix,
            user2id=user2id,
            id2item=id2item,
            k=k,
            recommend_fn=model.predict,
        )

        precision = precision_at_k(
            test_true=test_true,
            user_item_matrix=user_item_matrix,
            user2id=user2id,
            id2item=id2item,
            k=k,
            recommend_fn=model.predict,
        )

        ndcg = ndcg_at_k(
            test_true=test_true,
            user_item_matrix=user_item_matrix,
            user2id=user2id,
            id2item=id2item,
            k=k,
            recommend_fn=model.predict,
        )

        res[f"recall@{k}"] = recall
        res[f"precision@{k}"] = precision
        res[f"ndcg@{k}"] = ndcg

    return res

In [4]:
rows = []

models = [
    (
        "iALS",
        iALS(
            factors=32,
            regularization=0.08,
            iterations=10,
            random_state=42,
            num_threads=0,
        ),
    ),
    (
        "WeightedALS",
        WeightedALS(
            factors=32,
            regularization=0.08,
            iterations=10,
            alpha=40.0,
            random_state=42,
            num_threads=0,
        ),
    ),
    (
        "BPR",
        BPR(
            factors=32,
            regularization=0.01,
            iterations=30,
            learning_rate=0.05,
            random_state=42,
            num_threads=0,
        ),
    ),
]

for name, model in models:
    row = eval_model(model, name)
    rows.append(row)

pd.DataFrame(rows).sort_values("recall@10", ascending=False).reset_index(drop=True)

C:\Users\egorg\PycharmProjects\check5\checkpoint-5-bogdan-egor\venv\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 30/30 [00:05<00:00,  5.58it/s, train_auc=95.10%, skipped=6.27%]


,model,recall@10,precision@10,ndcg@10,recall@20,precision@20,ndcg@20
0,WeightedALS,0.001615,0.037470,0.038138,0.002987,0.034659,0.035976
1,iALS,0.001511,0.035060,0.036431,0.002762,0.032048,0.033807
2,BPR,0.001175,0.027269,0.028919,0.002110,0.024478,0.026372
